In [20]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
import xgboost as xgb

train = pd.read_csv('train.csv')

y = train['Churn'].map({'Yes': 1, 'No': 0})
X = train.drop(['Churn', 'id'], axis=1)
test_file = pd.read_csv('test.csv')
new_test = test_file.drop(['id'], axis=1)

# 1. Split into train and temporary test (80/20)
X_train_temp, X_test, y_train_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Split train_temp into train and validation (75/25 of 80% = 60/20)
X_train, X_val, y_train, y_val = train_test_split(X_train_temp, y_train_temp, test_size=0.25, random_state=42)

# Identify numeric and categorical columns
numeric_cols = [cname for cname in X_train.columns if X_train[cname].dtype in ['int64', 'float64']]
categorical_cols = [cname for cname in X_train.columns if X_train[cname].dtype == "object"]

# Numerical transformer
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# Categorical transformer
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Bundle preprocessing for both types of data
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# Define the model
#model = LogisticRegression()
model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=1,
    n_estimators=5000,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.8,
    early_stopping_rounds=5
)

# Bundle preprocessing and modeling code in a pipeline
my_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                              ('model', model)
                             ])

""" # Do not fit the pipeline before GridSearchCV — grid search will fit during CV
param_grid = {
    'model__max_depth': [3, 5, 7],
    'model__learning_rate': [0.1, 0.01, 0.05],
    'model__n_estimators': [100, 300],
    'model__subsample': [0.8, 1.0]
}

# 4. Set up GridSearchCV
# cv=5 performs 5-fold cross-validation
grid_search = GridSearchCV(
    estimator=my_pipeline,
    param_grid=param_grid,
    scoring='accuracy',
    n_jobs=-1,  # Use all available cores for parallel processing
    cv=5,
    verbose=1
)

# 5. Fit the model to find the best parameters. Pass eval_set and early stopping to the XGB fit via fit params
grid_search.fit(
    X_train_sub, y_train_sub,
    model__eval_set=[(X_val, y_val)],
    model__early_stopping_rounds=5,
    model__verbose=False
)

# 6. Retrieve the best parameters and score
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best CV Score: {grid_search.best_score_:.4f}")

# 7. Evaluate on test data
best_model = grid_search.best_estimator_
test_accuracy = best_model.score(X_test, y_test)
print(f"Test Accuracy: {test_accuracy:.4f}")
\my_pipeline.fit(
    X_train, y_train,
    model__eval_set=[(X_test, y_test)],
)
"""
X_val_processed = preprocessor.fit(X_train).transform(X_val)
my_pipeline.fit(
    X_train, y_train,
    model__eval_set=[(X_val_processed, y_val)],
)
preds = my_pipeline.predict_proba(new_test)
print(preds[:, 1])
print(y_test)
score = my_pipeline.score(X_test, y_test)
print(score)

output = pd.DataFrame({'id': test_file.id,
                       'Churn': preds[:, 1]})
output.to_csv('submission.csv', index=False)



<>:99: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
<>:99: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
/var/folders/yj/ml55_rvj7cd8sf2jswz70f300000gn/T/ipykernel_86655/4202360778.py:99: SyntaxWarning: "\m" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\m"? A raw string is also an option.
  \my_pipeline.fit(


[0]	validation_0-logloss:0.50050
[1]	validation_0-logloss:0.47545
[2]	validation_0-logloss:0.45577
[3]	validation_0-logloss:0.43948
[4]	validation_0-logloss:0.42571
[5]	validation_0-logloss:0.41424
[6]	validation_0-logloss:0.40403
[7]	validation_0-logloss:0.39557
[8]	validation_0-logloss:0.38812
[9]	validation_0-logloss:0.38192
[10]	validation_0-logloss:0.37610
[11]	validation_0-logloss:0.37125
[12]	validation_0-logloss:0.36688
[13]	validation_0-logloss:0.36303
[14]	validation_0-logloss:0.35942
[15]	validation_0-logloss:0.35656
[16]	validation_0-logloss:0.35358
[17]	validation_0-logloss:0.35130
[18]	validation_0-logloss:0.34929
[19]	validation_0-logloss:0.34705
[20]	validation_0-logloss:0.34530
[21]	validation_0-logloss:0.34385
[22]	validation_0-logloss:0.34229
[23]	validation_0-logloss:0.34111
[24]	validation_0-logloss:0.34004
[25]	validation_0-logloss:0.33888
[26]	validation_0-logloss:0.33775
[27]	validation_0-logloss:0.33679
[28]	validation_0-logloss:0.33613
[29]	validation_0-loglos